# Notebook pilote — Iran strikes cluster (28 fév 2026)

**Objectif** : valider la stack technique (Dune API + CLOB/Gamma APIs + DuckDB + polars) en reconstituant le cluster Bubblemaps de 7 wallets sur le marché "US strikes Iran by February 28, 2026?" (~$90M volume, résolu YES).

**Ground truth** : 6 wallets cluster Bubblemaps + Magamyman. Tous fresh wallets (joined 17-27 fév 2026), concentration ~100% sur ce marché, profit combiné ~$1.56M.

**Decision gate** : si on retrouve ≥ 5/7 wallets avec les features Niveau A (fresh + concentré + pré-event), la stack est validée pour la phase D.

---

## Partie 1 — Setup et chargement ground truth

### 1.1 Imports et configuration

On utilise polars (pas pandas) pour la perf, DuckDB pour le SQL analytique local, et httpx pour les appels API avec contrôle fin des timeouts et retries. Les clés API sont chargées depuis `.env` via python-dotenv.

In [1]:
import os
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import polars as pl
import duckdb
import httpx
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from dotenv import load_dotenv

# Chemins projet
ROOT = Path("..").resolve()
GT_DIR = ROOT / "data" / "ground_truth"
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Charger .env (DUNE_API_KEY requis pour Partie 2)
load_dotenv(ROOT / ".env")
DUNE_API_KEY = os.getenv("DUNE_API_KEY", "")
if not DUNE_API_KEY:
    print("⚠ DUNE_API_KEY manquante — Partie 2 (Dune) sera skippée, fallback CLOB API")
else:
    print(f"✓ DUNE_API_KEY chargée ({DUNE_API_KEY[:8]}...)")

# Style plots
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 5)

print(f"✓ Setup OK — polars {pl.__version__}, duckdb {duckdb.__version__}")

✓ DUNE_API_KEY chargée (Qf8pexuk...)
✓ Setup OK — polars 1.40.0, duckdb 1.5.2


### 1.2 Constantes du marché Iran strikes

Identifiants techniques récupérés via l'API Gamma. Le marché est sur le **vanilla CTF Exchange** (negRisk=false), ce qui simplifie les queries — une seule table Dune à interroger.

In [2]:
# --- Identifiants du marché Iran strikes ---
MARKET = {
    "id": "1198423",
    "question": "US strikes Iran by February 28, 2026?",
    "condition_id": "0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20",
    "token_yes": "110790003121442365126855864076707686014650523258783405996925622264696084778807",
    "token_no": "10832696757358093775468120009000761778513405247768868107262967513475277652998",
    "neg_risk": False,
    "resolution": "Yes",
    "event_date": datetime(2026, 2, 28, 9, 31, tzinfo=timezone.utc),
    "volume_usd": 89_652_867,
}

# Période de backfill : 30 jours avant l'event pour capturer le buildup
BACKFILL_START = datetime(2026, 1, 29, tzinfo=timezone.utc)
BACKFILL_END = datetime(2026, 3, 1, tzinfo=timezone.utc)

print(f"Marché : {MARKET['question']}")
print(f"Condition ID : {MARKET['condition_id']}")
print(f"Résolu : {MARKET['resolution']} le {MARKET['event_date'].strftime('%Y-%m-%d %H:%M UTC')}")
print(f"Volume : ${MARKET['volume_usd']:,.0f}")
print(f"Backfill : {BACKFILL_START.date()} → {BACKFILL_END.date()}")
print(f"Exchange : {'Neg Risk' if MARKET['neg_risk'] else 'Vanilla CTF'}")

Marché : US strikes Iran by February 28, 2026?
Condition ID : 0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20
Résolu : Yes le 2026-02-28 09:31 UTC
Volume : $89,652,867
Backfill : 2026-01-29 → 2026-03-01
Exchange : Vanilla CTF


### 1.3 Chargement du ground truth

On charge les 3 CSVs existants et on filtre sur le cas Iran (case_id=2). Les wallets attendus sont les 6 du cluster Bubblemaps + Magamyman.

In [3]:
# Charger les CSVs ground truth
cases = pl.read_csv(GT_DIR / "cases.csv", truncate_ragged_lines=True)
wallets = pl.read_csv(GT_DIR / "wallets.csv", truncate_ragged_lines=True)
sharps = pl.read_csv(GT_DIR / "sharps_positive.csv", truncate_ragged_lines=True)

# Filtrer sur le cas Iran strikes (case_id = 2)
iran_cases = cases.filter(pl.col("case_id") == 2)
iran_wallets = wallets.filter(pl.col("case_id") == 2)

# Extraire les adresses connues (non vides)
gt_addresses = (
    iran_wallets
    .filter(pl.col("address").is_not_null() & (pl.col("address") != ""))
    .select("address", "cluster_name", "role", "notes")
)

print(f"=== Ground Truth Iran strikes (case_id=2) ===")
print(f"Lignes dans wallets.csv : {len(iran_wallets)}")
print(f"Adresses connues       : {len(gt_addresses)}")
print(f"Adresses manquantes    : {len(iran_wallets) - len(gt_addresses)}")
print()

# Liste des adresses à chercher dans les trades (lowercase pour matching)
GT_ADDRS = [a.lower() for a in gt_addresses["address"].to_list()]

print("Adresses cibles (lowercase) :")
for i, row in enumerate(gt_addresses.iter_rows(named=True)):
    addr = row["address"]
    note = row["notes"][:60] if row["notes"] else ""
    print(f"  {i+1}. {addr[:10]}...{addr[-6:]}  — {note}")

=== Ground Truth Iran strikes (case_id=2) ===
Lignes dans wallets.csv : 8
Adresses connues       : 7
Adresses manquantes    : 1

Adresses cibles (lowercase) :
  1. 0x1caA6a7a...4846a0  — 560 680 Yes shares @ ~$0.108. Profit ~$494 375. Plus gros ga
  2. 0xa4eb5222...acd010  — nothingeverhappens911. Lié à cluster Skoobidoobnj via dépôt 
  3. 0x3811e09b...478f61  — 
  4. 0xdde15ebd...2e02c5  — @Dicedicedice. 150 000 shares @ $0.20. Profit ~$120 000. Joi
  5. 0x56efadc9...4136db  — @Neodbs. $89K profit. 1 prediction. Joined Feb 26 2026. Enri
  6. 0x38745db2...149801  — @Planktonbets. $174K profit. 7 predictions. Joined Feb 17 20
  7. 0x4dfd481c...9f6e4a  — Magamyman. Actif depuis oct 2024. +$814K P&L (API). Premier 


---

## Partie 2 — Ingestion des trades

**Pivot** : l'endpoint CLOB `/trades` est désormais auth-only (401 sans clé API). On utilise la **Data API** (`data-api.polymarket.com/trades?user=<addr>`) qui reste publique et ne nécessite aucune authentification.

**Approche** : au lieu de pull tous les trades du marché puis filtrer, on query wallet par wallet (les 7 adresses ground truth), puis on filtre côté client sur le `conditionId` du marché Iran. C'est plus ciblé et suffisant pour le pilote.

Le marché est sur le vanilla CTF Exchange (negRisk=false). La Data API retourne directement `conditionId`, `side`, `size`, `price`, `timestamp`, `outcome` — pas besoin de mapper manuellement.

### 2.1 Helper — appels API avec retry et rate limiting

Fonction utilitaire pour tous les appels HTTP avec backoff exponentiel sur les 429 et logging du nombre d'appels.

In [4]:
class APIStats:
    """Compteur global d'appels API pour le logging en fin de partie."""
    def __init__(self):
        self.calls = 0
        self.errors = 0
        self.retries = 0

    def summary(self, label: str) -> str:
        return f"[{label}] {self.calls} calls, {self.retries} retries, {self.errors} errors"

stats_clob = APIStats()
stats_gamma = APIStats()
stats_dune = APIStats()


def api_get(url: str, params: dict = None, stats: APIStats = None,
            sleep_s: float = 0.12, max_retries: int = 3, timeout: float = 30.0) -> dict | list | None:
    """GET request with retry + backoff on 429. Returns parsed JSON or None on failure."""
    if stats:
        stats.calls += 1
    for attempt in range(max_retries):
        try:
            time.sleep(sleep_s)
            with httpx.Client(timeout=timeout) as client:
                resp = client.get(url, params=params)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait = 2 ** (attempt + 1)
                print(f"  429 rate-limited, retrying in {wait}s...")
                if stats:
                    stats.retries += 1
                time.sleep(wait)
            else:
                print(f"  HTTP {resp.status_code} on {url}")
                if stats:
                    stats.errors += 1
                return None
        except httpx.TimeoutException:
            print(f"  Timeout on {url}, attempt {attempt+1}/{max_retries}")
            if stats:
                stats.retries += 1
            time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  Error: {e}")
            if stats:
                stats.errors += 1
            return None
    print(f"  Failed after {max_retries} retries: {url}")
    if stats:
        stats.errors += 1
    return None

print("✓ Helper API prêt")

✓ Helper API prêt


### 2.2 Ingestion via Data API (wallet par wallet)

L'endpoint `GET /trades?user=<address>` retourne tous les trades d'un wallet, paginés. On itère sur chaque adresse ground truth, puis on filtre côté client sur le `conditionId` du marché Iran. Rate limit : sleep 1s entre wallets pour rester safe.

In [ ]:
DATA_API = "https://data-api.polymarket.com"
IRAN_CONDITION_ID = MARKET["condition_id"]

stats_data = APIStats()


def fetch_wallet_trades(address: str, limit: int = 500) -> list[dict]:
    """Fetch all trades for a wallet via Data API, with pagination via offset."""
    all_trades = []
    offset = 0

    while True:
        data = api_get(
            f"{DATA_API}/trades",
            params={"user": address, "limit": limit, "offset": offset},
            stats=stats_data, sleep_s=1.0, timeout=30.0
        )
        if data is None:
            print(f"    ✗ Erreur sur offset={offset}, arrêt pour ce wallet")
            break

        trades = data if isinstance(data, list) else data.get("data", [])
        if not trades:
            break

        all_trades.extend(trades)
        offset += len(trades)

        # Si on a reçu moins que limit, c'est la dernière page
        if len(trades) < limit:
            break

    return all_trades


# --- Ingestion wallet par wallet ---
print(f"Ingestion via Data API — {len(GT_ADDRS)} wallets ground truth")
print(f"Filtre conditionId : {IRAN_CONDITION_ID}")
print()

all_iran_trades = []

for addr in GT_ADDRS:
    short = addr[:8] + "..." + addr[-4:]
    print(f"  {short} : ", end="")

    wallet_trades = fetch_wallet_trades(addr)
    print(f"{len(wallet_trades)} trades totaux", end="")

    # Filtrer sur le marché Iran (conditionId match)
    iran_trades = [t for t in wallet_trades if t.get("conditionId") == IRAN_CONDITION_ID]
    print(f" → {len(iran_trades)} sur marché Iran")

    # Ajouter l'adresse wallet dans chaque trade pour le tracking
    for t in iran_trades:
        t["wallet"] = addr.lower()

    all_iran_trades.extend(iran_trades)

print(f"\n{'='*50}")
print(f"Total trades Iran cluster : {len(all_iran_trades)}")
print(stats_data.summary("Data API"))

### 2.3 Normalisation et stockage en Parquet

La Data API retourne des colonnes standardisées : `proxyWallet`, `side`, `size`, `price`, `timestamp` (epoch seconds), `conditionId`, `outcome`, `transactionHash`, `title`, `name`. On normalise les types et on stocke en Parquet.

In [ ]:
if all_iran_trades:
    # Inspecter le schéma
    sample = all_iran_trades[0]
    print("Clés disponibles dans un trade Data API :")
    for k, v in sample.items():
        print(f"  {k}: {type(v).__name__} = {str(v)[:80]}")
else:
    print("⚠ Aucun trade récupéré — vérifier les adresses et le conditionId")

In [ ]:
if all_iran_trades:
    df = pl.DataFrame(all_iran_trades)

    # Sélectionner et renommer les colonnes utiles
    keep_cols = {
        "wallet": "wallet",
        "proxyWallet": "proxy_wallet",
        "side": "side",
        "size": "size",
        "price": "price",
        "timestamp": "timestamp",
        "outcome": "outcome",
        "outcomeIndex": "outcome_index",
        "conditionId": "condition_id",
        "transactionHash": "tx_hash",
        "title": "title",
        "name": "username",
        "asset": "asset_id",
    }
    # Garder seulement les colonnes présentes
    rename_map = {k: v for k, v in keep_cols.items() if k in df.columns}
    df = df.select(list(rename_map.keys())).rename(rename_map)

    # Cast types
    df = df.with_columns([
        pl.col("size").cast(pl.Float64),
        pl.col("price").cast(pl.Float64),
        pl.col("wallet").str.to_lowercase(),
    ])

    # Convertir timestamp epoch → datetime
    if df["timestamp"].dtype in (pl.Int64, pl.Float64):
        df = df.with_columns(
            pl.from_epoch(pl.col("timestamp"), time_unit="s").alias("datetime")
        )
    elif df["timestamp"].dtype == pl.String:
        df = df.with_columns(
            pl.from_epoch(pl.col("timestamp").cast(pl.Int64), time_unit="s").alias("datetime")
        )

    # Calculer size en USDC (size = nb shares, USDC = size * price)
    df = df.with_columns(
        (pl.col("size") * pl.col("price")).alias("size_usd")
    )

    # Trier par timestamp
    df = df.sort("datetime")

    # Sauvegarder
    parquet_path = RAW_DIR / "iran_trades.parquet"
    df.write_parquet(parquet_path)

    print(f"✓ {len(df)} trades normalisés")
    print(f"  Colonnes : {df.columns}")
    print(f"  Période  : {df['datetime'].min()} → {df['datetime'].max()}")
    print(f"  Sauvegardé : {parquet_path} ({parquet_path.stat().st_size / 1024:.0f} KB)")
    print()
    print(df.head(5))
else:
    df = pl.DataFrame()
    print("⚠ DataFrame vide — pas de trades Iran trouvés")

### 2.4 Checkpoint Partie 2

Vérification rapide : combien de trades récupérés, volume total, et les adresses ground truth sont-elles présentes ?

In [ ]:
# Checkpoint Partie 2
if len(df) > 0:
    # Volume total
    total_vol = df["size_usd"].sum() if "size_usd" in df.columns else df["size"].sum()
    total_shares = df["size"].sum()
    print(f"Volume total      : {total_shares:,.0f} shares (~${total_vol:,.0f} USDC)")
    print(f"Trades uniques    : {len(df)}")

    # Wallets présents
    wallets_in_data = df["wallet"].unique().to_list()
    print(f"\nWallets dans les données : {len(wallets_in_data)}")

    found = 0
    for addr in GT_ADDRS:
        present = addr in wallets_in_data
        if present:
            found += 1
            n = len(df.filter(pl.col("wallet") == addr))
            vol = df.filter(pl.col("wallet") == addr)["size_usd"].sum()
            status = f"✓ TROUVÉ — {n} trades, ~${vol:,.0f}"
        else:
            status = "✗ ABSENT"
        print(f"  {addr[:10]}...{addr[-4:]} : {status}")

    print(f"\n{'='*50}")
    print(f"CHECKPOINT 2 : {found}/{len(GT_ADDRS)} wallets ground truth retrouvés")
    print(f"{'='*50}")
else:
    print("⚠ Pas de données — vérifier Partie 2.2")

---

## Partie 3 — Enrichissement métadonnées

On complète les trades avec trois sources :
1. **Gamma API** — metadata marché (question exacte, outcomes, résolution, dates)
2. **Data API** — profils et historiques des wallets cibles
3. **CLOB API** — prix OHLC historiques pour tracer la timeline du marché

### 3.1 Metadata marché via Gamma API

On récupère les détails complets du marché pour vérifier la cohérence avec nos constantes (question, resolution status, dates).

In [9]:
# Gamma API — metadata du marché
market_meta = api_get(
    f"https://gamma-api.polymarket.com/markets/{MARKET['id']}",
    stats=stats_gamma, sleep_s=0.5
)

if market_meta:
    # Sauvegarder le JSON complet
    meta_path = RAW_DIR / "iran_metadata.json"
    with open(meta_path, "w") as f:
        json.dump(market_meta, f, indent=2)

    print("=== Metadata marché (Gamma API) ===")
    for key in ["question", "conditionId", "outcomes", "outcomePrices",
                "volume", "closed", "umaResolutionStatus", "negRisk",
                "startDate", "endDate", "closedTime"]:
        val = market_meta.get(key, "N/A")
        print(f"  {key}: {val}")

    print(f"\n✓ Sauvegardé : {meta_path}")
else:
    print("✗ Impossible de récupérer les metadata Gamma")

=== Metadata marché (Gamma API) ===
  question: US strikes Iran by February 28, 2026?
  conditionId: 0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20
  outcomes: ["Yes", "No"]
  outcomePrices: ["1", "0"]
  volume: 89652867.363489
  closed: True
  umaResolutionStatus: resolved
  negRisk: False
  startDate: 2026-01-19T20:34:32.385Z
  endDate: 2026-01-31T00:00:00Z
  closedTime: 2026-02-28 09:31:17+00

✓ Sauvegardé : /Users/gabsav/Documents/Polycasquette/Code/data/raw/iran_metadata.json


### 3.2 Prix historiques via Gamma API

L'endpoint CLOB `/prices-history` a renvoyé vide sur ce marché résolu. On utilise les prix d'entrée de nos trades (colonne `price`) pour reconstituer la timeline de prix. En complément, l'endpoint Gamma `/markets/{id}` contient `outcomePrices` (prix final) qu'on a déjà récupéré en 3.1.

In [ ]:
# Reconstituer une timeline de prix à partir des trades (prix d'entrée par heure)
if len(df) > 0 and "datetime" in df.columns and "price" in df.columns:
    df_prices = (
        df
        .with_columns(pl.col("datetime").dt.truncate("1h").alias("hour"))
        .group_by("hour")
        .agg([
            pl.col("price").mean().alias("price_mean"),
            pl.col("price").min().alias("price_min"),
            pl.col("price").max().alias("price_max"),
            pl.col("size_usd").sum().alias("volume_usd"),
            pl.len().alias("n_trades"),
        ])
        .sort("hour")
    )

    prices_path = RAW_DIR / "iran_prices.parquet"
    df_prices.write_parquet(prices_path)

    print(f"✓ Timeline prix reconstituée : {len(df_prices)} points horaires")
    print(f"  Période : {df_prices['hour'].min()} → {df_prices['hour'].max()}")
    print(f"  Sauvegardé : {prices_path}")
    print()
    print(df_prices.tail(10))
else:
    df_prices = pl.DataFrame()
    print("⚠ Pas de données pour reconstituer les prix")

### 3.3 Log API Partie 3

In [ ]:
print("=== Log API cumulé ===")
print(stats_clob.summary("CLOB"))
print(stats_gamma.summary("Gamma"))
print(stats_data.summary("Data API"))
print(stats_dune.summary("Dune"))

# Fichiers produits
print("\n=== Fichiers produits ===")
for p in sorted(RAW_DIR.glob("iran_*")):
    print(f"  {p.name} ({p.stat().st_size / 1024:.0f} KB)")

---

## Partie 4 — Validation et analyse

On vérifie que les wallets ground truth sont bien dans les données, on calcule leurs métriques clés (volume, timing, concentration), et on produit la visualisation timeline.

### 4.1 Table de vérification par wallet

Pour chaque wallet ground truth, on extrait : nombre de trades, volume cumulé, premier et dernier trade, et on compare avec les dates attendues (joined 17-27 fév 2026).

In [ ]:
# Charger les données depuis le Parquet (point de reprise si on relance)
parquet_path = RAW_DIR / "iran_trades.parquet"
if parquet_path.exists():
    df = pl.read_parquet(parquet_path)
    print(f"Trades chargés : {len(df)}")
    print(f"Colonnes : {df.columns}")
else:
    print("⚠ iran_trades.parquet n'existe pas — relancer la Partie 2")
    df = pl.DataFrame()

In [ ]:
# Table de vérification : chaque wallet ground truth vs données
event_dt = MARKET["event_date"]
results = []

for addr in GT_ADDRS:
    wallet_trades = df.filter(pl.col("wallet") == addr) if len(df) > 0 else pl.DataFrame()
    n_trades = len(wallet_trades)

    if n_trades > 0 and "datetime" in wallet_trades.columns:
        first_ts = wallet_trades["datetime"].min()
        last_ts = wallet_trades["datetime"].max()
        # time_to_event en heures (depuis le dernier trade jusqu'à l'event)
        try:
            last_aware = last_ts.replace(tzinfo=timezone.utc) if last_ts.tzinfo is None else last_ts
            hours_before = (event_dt - last_aware).total_seconds() / 3600
        except Exception:
            hours_before = None
    else:
        first_ts = last_ts = None
        hours_before = None

    vol = wallet_trades["size_usd"].sum() if n_trades > 0 and "size_usd" in wallet_trades.columns else 0
    shares = wallet_trades["size"].sum() if n_trades > 0 and "size" in wallet_trades.columns else 0

    # Nom depuis ground truth
    gt_row = iran_wallets.filter(pl.col("address").str.to_lowercase() == addr)
    name = gt_row["notes"][0][:40] if len(gt_row) > 0 and gt_row["notes"][0] else addr[:12]

    results.append({
        "wallet": addr[:10] + "..." + addr[-4:],
        "name": name,
        "found": n_trades > 0,
        "n_trades": n_trades,
        "shares": round(shares, 0),
        "volume_usd": round(vol, 2),
        "first_trade": str(first_ts)[:19] if first_ts else "—",
        "last_trade": str(last_ts)[:19] if last_ts else "—",
        "hours_before_event": round(hours_before, 1) if hours_before is not None else None,
    })

df_check = pl.DataFrame(results)
print("=== Table de vérification wallets ground truth ===")
print(df_check)

### 4.2 Timeline visuelle

Visualisation de tous les trades sur le marché Iran. Les wallets ground truth sont mis en évidence en couleur. Ligne verticale rouge = date de l'event (28 fév 2026 09:31 UTC). Cela permet de voir visuellement le buildup pré-event.

In [ ]:
if len(df) > 0 and "datetime" in df.columns and "size_usd" in df.columns:
    fig, ax = plt.subplots(figsize=(16, 6))

    # Couleurs par wallet ground truth
    colors = ["#e74c3c", "#2ecc71", "#3498db", "#f39c12", "#9b59b6", "#1abc9c", "#e67e22"]

    for i, addr in enumerate(GT_ADDRS):
        wt = df.filter(pl.col("wallet") == addr)
        if len(wt) > 0:
            short = addr[:6] + "..." + addr[-4:]
            # Récupérer le username si disponible
            usernames = wt["username"].unique().to_list() if "username" in wt.columns else []
            label = usernames[0] if usernames and usernames[0] else short

            ax.scatter(
                wt["datetime"].to_list(), wt["size_usd"].to_list(),
                s=50, alpha=0.8, color=colors[i % len(colors)],
                label=label, edgecolors="black", linewidths=0.3, zorder=5
            )

    # Ligne verticale = event date
    ax.axvline(x=event_dt, color="red", linestyle="--", linewidth=2, label="Event (28 fév 09:31 UTC)")

    ax.set_xlabel("Date")
    ax.set_ylabel("Trade size (USDC)")
    ax.set_title(f"Timeline trades cluster Iran — {MARKET['question']}")
    ax.legend(fontsize=9, loc="upper left")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(RAW_DIR / "iran_timeline.png", dpi=150)
    plt.show()
    print("✓ Graphique sauvegardé : data/raw/iran_timeline.png")
else:
    print("⚠ Pas assez de données pour le graphique")

### 4.3 Verdict final

Check final : les 3 questions clés auxquelles le pilote doit répondre.

1. Les wallets ground truth sont-ils dans les données ?
2. Quel est leur volume cumulé ?
3. Est-ce que leur timing matche la ground truth (pré-event < 48h) ?

In [ ]:
# === VERDICT FINAL ===
print("=" * 60)
print("VERDICT PILOTE IRAN STRIKES")
print("=" * 60)

if len(df_check) > 0:
    n_found = df_check.filter(pl.col("found") == True).height
    n_total = len(df_check)
    total_vol = df_check["volume_usd"].sum()
    total_shares = df_check["shares"].sum()

    # Q1 : wallets retrouvés ?
    print(f"\n1. Wallets retrouvés : {n_found}/{n_total}")
    if n_found >= 5:
        print("   ✓ PASS — ≥ 5/7 wallets trouvés dans les données")
    elif n_found >= 3:
        print("   ~ PARTIAL — 3-4 wallets trouvés, investiguer les manquants")
    else:
        print("   ✗ FAIL — < 3 wallets trouvés")

    # Q2 : volume cumulé
    print(f"\n2. Volume cumulé wallets GT : {total_shares:,.0f} shares (~${total_vol:,.0f} USDC)")

    # Q3 : timing pré-event
    pre_event = df_check.filter(
        pl.col("hours_before_event").is_not_null()
        & (pl.col("hours_before_event") > 0)
        & (pl.col("hours_before_event") < 48)
    )
    print(f"\n3. Wallets avec dernier trade < 48h avant event : {pre_event.height}/{n_found}")
    if n_found > 0 and pre_event.height >= n_found * 0.7:
        print("   ✓ PASS — majorité des wallets tradent pré-event < 48h")
    elif n_found > 0:
        print("   ~ Résultat mixte — vérifier les timestamps")
    else:
        print("   ✗ Aucun wallet trouvé")

    # Décision globale
    print(f"\n{'=' * 60}")
    if n_found >= 5:
        print("DECISION : ✓ Pipeline validé — passer aux expériences C1/C2/C3")
    elif n_found >= 3:
        print("DECISION : ~ Pipeline partiellement validé — investiguer")
    else:
        print("DECISION : ✗ Pipeline échoue — vérifier les sources de données")
    print("=" * 60)
else:
    print("⚠ Pas de données de vérification disponibles")

# Log final
print(f"\n=== Log API total ===")
print(stats_clob.summary("CLOB"))
print(stats_gamma.summary("Gamma"))
print(stats_data.summary("Data API"))
print(stats_dune.summary("Dune"))